In [ ]:
from acm.hod import CutskyHOD
from acm import setup_logging
from pyrecon.utils import sky_to_cartesian
import numpy as np
from pathlib import Path
import pandas
from cosmoprimo.fiducial import AbacusSummit
from mockfactory.desi import build_tiles_for_fa, apply_fiber_assignment
from mpi4py import MPI
import fitsio
from mpytools import Catalog

mpicomm = MPI.COMM_WORLD
setup_logging(level=('info' if mpicomm.rank == 0 else 'warning'))

In [ ]:
def get_hod_params(nrows=None):
    """Some example HOD parameters."""
    hod_dir = Path(f'/pscratch/sd/e/epaillas/emc/hod_params/yuan23/')
    hod_fn = hod_dir / f'hod_params_yuan23_c000.csv'
    df = pandas.read_csv(hod_fn, delimiter=',')
    df.columns = df.columns.str.strip()
    df.columns = list(df.columns.str.strip('# ').values)
    return df.to_dict('list')

# redshifts of the snapshots that will be used to build the cutsky
snapshots = [0.5]

# redshift range (in the cutsky) that will be covered by each snapshot
zranges = [(0.4, 0.6)]

# fiducial cosmology for the redshift-distance relation and RSD
cosmo = AbacusSummit(0)

# read example HOD parameters
hod_params = get_hod_params()

# initialize class
nz_filename='/global/cfs/cdirs/desi/survey/catalogs/Y1/LSS/iron/LSScats/v1.5/LRG_NGC_nz.txt'

cutsky = CutskyHOD(varied_params=hod_params.keys(),
                zranges=zranges, snapshots=snapshots,
                cosmo_idx=0, phase_idx=0,
                load_existing_hod=True)

# sample HOD parameters and build the cutsky mock
hod = {key: hod_params[key][30] for key in hod_params.keys()}
cutsky.sample_hod(hod, nthreads=1, region='NGC', release='Y1', target_nz_filename=nz_filename)
cutsky.apply_angular_mask(region='NGC', release='Y1', npasses=None, program='dark')
nz_filename='/global/cfs/cdirs/desi/survey/catalogs/Y1/LSS/iron/LSScats/v1.5/LRG_NGC_nz.txt'
cutsky.apply_radial_mask(nz_filename=nz_filename)

In [ ]:
cutsky = CutskyHOD(varied_params=hod_params.keys(),
                        zranges=zranges, snapshots=snapshots,
                        cosmo_idx=0, phase_idx=0,
                        load_existing_hod=True)

In [ ]:
cutsky.catalog

In [ ]:
# reduce fp for testing
mask = (cutsky.catalog['RA']<205) &(cutsky.catalog['RA']> 175) & (cutsky.catalog['DEC']>35) & (cutsky.catalog['DEC']<45)

for key in cutsky.catalog.keys():
    cutsky.catalog[key] = cutsky.catalog[key][mask]
 

In [ ]:
# To show on a healpix map 
# to install CRStools:
# git clone https://github.com/4most-crs/4MOST_CRS_tools.git 
# cd 4MOST_CRS_tools
# python -m pip install -e . --user

from CRStools import utils 
hpmap = utils.create_hp_map(cutsky.catalog['RA'], cutsky.catalog['DEC'], nside=256)
utils.plot_moll(hpmap, figsize=(10,8), desi_footprint=True)


In [ ]:
#Plot use skyproj: python -m pip install skyproj
import skyproj


cutsky.run_FA_for_single_tracer(release='Y1', program='dark', npasses=1, use_sky_targets=False, 
                                 seed=None, preload_sky_targets=False, plot_output=True, mpicomm=None)

In [ ]:
import skyproj
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
# sp.draw_des(label='DES', edgecolor='k')
sp = skyproj.DESSkyproj(ax=ax, extent=[178,209, 33, 48], fontsize=8)

sp.scatter(cutsky.catalog['RA'], cutsky.catalog['DEC'], s=0.001, c='r')
# sp.scatter(cutsky.catalog_rd['RA'][cutsky.catalog_rd['AVAILABLE']], cutsky.catalog_rd['DEC'][cutsky.catalog_rd['AVAILABLE']], s=0.01,c='r') 
# sp.scatter(cutsky.catalog['RA'][cutsky.catalog['AVAILABLE']], cutsky.catalog['DEC'][cutsky.catalog['AVAILABLE']], s=0.01,c='b') 
sp.scatter(cutsky.catalog['RA'][cutsky.catalog['NUMOBS'] >0], cutsky.catalog['DEC'][cutsky.catalog['NUMOBS'] >0], s=0.01,c='b') 
# sp.scatter(cutsky.catalog['RA'][cutsky.catalog['OBS_PASS'].T[0]], cutsky.catalog['DEC'][cutsky.catalog['OBS_PASS'].T[0]], s=0.01,c='b') 

sp.scatter(0,0, s=10,c='r', label='cutsky mock') 
sp.scatter(0,0, s=10,c='b', label='Observed targets') 



sp.legend()
fig.tight_layout()

In [ ]:
# Load results from test_fa.py
import os 
pscratch = os.environ["PSCRATCH"]

cutsky_fa = Catalog.load(f'{pscratch}/cutsky_FA.fits.npy')

In [ ]:
cutsky_fa['RA'].cmax()

In [ ]:
import skyproj
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
# sp.draw_des(label='DES', edgecolor='k')
sp = skyproj.DESSkyproj(ax=ax, extent=[178,209, 33, 48], fontsize=8)


sp.scatter(cutsky_fa['RA'][cutsky_fa['AVAILABLE']], cutsky_fa['DEC'][cutsky_fa['AVAILABLE']], s=0.01,c='r') 
sp.scatter(cutsky_fa['RA'][cutsky_fa['OBS_PASS'].T[0]], cutsky_fa['DEC'][cutsky_fa['OBS_PASS'].T[0]], s=0.01,c='b') 
sp.scatter(cutsky_fa['RA'][cutsky_fa['OBS_PASS'].T[1]], cutsky_fa['DEC'][cutsky_fa['OBS_PASS'].T[1]], s=0.01,c='green') 

# sp.scatter(cutsky_fa['RA'][cutsky_fa['NUMOBS']>0], cutsky_fa['DEC'][cutsky_fa['NUMOBS']>0], s=0.01,c='b') 

sp.scatter(0,0, s=10,c='r', label='cutsky mock') 
sp.scatter(0,0, s=10,c='b', label='Assigned targets pass 1') 
sp.scatter(0,0, s=10,c='g', label='Assigned targets pass 2') 



sp.legend(fontsize=9)
fig.tight_layout()

In [ ]:
import skyproj
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
# sp.draw_des(label='DES', edgecolor='k')
sp = skyproj.DESSkyproj(ax=ax, extent=[178,209, 33, 48], fontsize=8)


sp.scatter(cutsky_fa['RA'][cutsky_fa['AVAILABLE']], cutsky_fa['DEC'][cutsky_fa['AVAILABLE']], s=0.01,c='r') 


sp.scatter(cutsky_fa['RA'][cutsky_fa['NUMOBS']>0], cutsky_fa['DEC'][cutsky_fa['NUMOBS']>0], s=0.01,c='b') 

sp.scatter(0,0, s=10,c='r', label='cutsky mock') 
sp.scatter(0,0, s=10,c='b', label='All assigned targets') 


sp.legend()
fig.tight_layout()